# 🛠️ Notebook 2: Stack Overflow — Bad → Good → Best

Three versions of the same feature set, each one fixing a specific pain from the previous.

| Version | Style | What it fixes |
|---------|-------|---------------|
| 🚫 Bad  | "God class" — one object owns users, questions, answers, votes, rep | Baseline — shows the pain |
| ✅ Good | Proper entities — `User`, `Post`, `Question`, `Answer`, `vote()`     | Separation of concerns, testable |
| 🌟 Best | Same entities + enums, statuses, guarded votes, typed dataclasses    | Readable, hard to misuse, easy to extend |

Every version is fully runnable — you can scroll, compare, and run each cell on its own.


## 🛠️ Setup

```bash
cd 07-object-oriented-design/stack-overflow
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🚫 Version 1 — The "god class" (anti-pattern)

Everything — users, posts, votes, reputation — lives on one `BadSO` class.
It works for a demo, but it's already painful:

- Data is stored in parallel dictionaries that must stay in sync (bug magnet).
- Business rules (who can vote, rep math) are buried inside unrelated CRUD code.
- You can't test "voting" without dragging the whole object along.
- Adding *comments* or *tags* means touching this one giant class again.

This is the baseline. Don't do this in real code.


In [1]:
class BadSO:
    """One class owns the whole world. Don't do this in real code."""
    def __init__(self):
        self.users = {}        # user_id -> {'name':..., 'rep':...}
        self.questions = {}    # qid -> {'author':..., 'title':..., 'body':..., 'votes':{}, 'answers':[]}
        self.answers = {}      # aid -> {'author':..., 'qid':..., 'body':..., 'votes':{}}
        self._next_id = 1

    def _new_id(self):
        self._next_id += 1
        return self._next_id

    def add_user(self, name):
        uid = self._new_id()
        self.users[uid] = {'name': name, 'rep': 0}
        return uid

    def ask(self, user_id, title, body):
        qid = self._new_id()
        self.questions[qid] = {'author': user_id, 'title': title, 'body': body, 'votes': {}, 'answers': []}
        return qid

    def answer(self, user_id, qid, body):
        aid = self._new_id()
        self.answers[aid] = {'author': user_id, 'qid': qid, 'body': body, 'votes': {}}
        self.questions[qid]['answers'].append(aid)
        return aid

    def vote(self, voter_id, post_id, direction):
        # Which bucket is it in?  (Already smelly: we have to guess.)
        if post_id in self.questions:
            post = self.questions[post_id]; per_upvote = 5
        elif post_id in self.answers:
            post = self.answers[post_id];   per_upvote = 10
        else:
            raise KeyError(post_id)
        # No self-vote check!  Bug waiting to happen.
        prev = post['votes'].get(voter_id, 0)
        post['votes'][voter_id] = direction
        delta = direction - prev
        author = self.users[post['author']]
        if delta > 0:
            author['rep'] += per_upvote * delta
        elif delta < 0:
            author['rep'] += -2 * (-delta)

# Try it
so = BadSO()
u_ada   = so.add_user('Ada')
u_grace = so.add_user('Grace')
qid  = so.ask(u_ada, 'What is OOD?', 'Explain please.')
aid  = so.answer(u_grace, qid, 'Start with SOLID.')
so.vote(u_grace, qid, +1)
so.vote(u_ada,   aid, +1)
# Notice: Ada could have upvoted her OWN question - the bad class doesn't stop her.
so.vote(u_ada,   qid, +1)
print('Ada   rep =', so.users[u_ada]['rep'])     # inflated by self-vote
print('Grace rep =', so.users[u_grace]['rep'])


Ada   rep = 10
Grace rep = 10


### What hurts in the bad version

1. **Guess-the-bucket** — `vote()` has to check which dict the id is in. Types would have told us.
2. **No self-vote guard** — Ada silently up-voted her own question.
3. **No `score` property** — every caller has to `sum(...)` the dict themselves.
4. **No comments, no tags, no accepted answer** — adding them means more fields on one bloated class.
5. **Tests can't isolate voting** — you always need the whole `BadSO`.

Let's fix all of that by giving each concept its own class.


## ✅ Version 2 — Proper classes

Now each noun gets its own class. Behavior lives next to the data it touches.
Reputation is applied by a *free function* `vote()` so the rule is in one place.


In [2]:
from dataclasses import dataclass
from typing import Optional, List
import itertools

@dataclass
class User:
    id: str
    name: str
    reputation: int = 0

_pid = itertools.count(1)

class Post:
    """Shared behavior: votes, comments, score."""
    def __init__(self, author: User, body: str):
        self.id = next(_pid)
        self.author = author
        self.body = body
        self.votes = {}          # user_id -> +1/-1
        self.comments: List[tuple] = []

    @property
    def score(self) -> int:
        return sum(self.votes.values())

    def comment(self, user: User, text: str) -> None:
        self.comments.append((user.id, text))

class Question(Post):
    def __init__(self, author, title, body, tags):
        super().__init__(author, body)
        self.title = title
        self.tags = list(tags)
        self.answers: List['Answer'] = []
        self.accepted: Optional['Answer'] = None

    def _add_answer(self, ans: 'Answer') -> None:
        self.answers.append(ans)

    def accept(self, ans: 'Answer') -> None:
        if ans not in self.answers:
            raise ValueError('not an answer to this question')
        if self.accepted is not None:
            raise ValueError('an answer is already accepted')
        self.accepted = ans
        ans.author.reputation += 15     # acceptance bonus

class Answer(Post):
    def __init__(self, author, question: Question, body):
        super().__init__(author, body)
        self.question = question
        question._add_answer(self)

def vote(post: Post, voter: User, direction: int) -> None:
    assert direction in (1, -1), 'direction must be +1 or -1'
    if voter.id == post.author.id:
        raise ValueError('cannot vote on your own post')
    prev = post.votes.get(voter.id, 0)
    post.votes[voter.id] = direction
    delta = direction - prev
    per_upvote = 10 if isinstance(post, Answer) else 5
    if delta > 0:
        post.author.reputation += per_upvote * delta
    elif delta < 0:
        post.author.reputation += -2 * (-delta)

# ---- walk-through ----
ada   = User('u1','Ada')
grace = User('u2','Grace')
bob   = User('u3','Bob')

q  = Question(ada, 'What is OOD?', 'I want to learn.', ['ood','design'])
a1 = Answer(grace, q, 'Start with SOLID principles ...')
a2 = Answer(bob,   q, 'Practice with the parking-lot problem.')

vote(q,  grace, +1)     # Ada  +5
vote(a1, ada,   +1)     # Grace +10
vote(a1, bob,   +1)     # Grace +10
vote(a2, ada,   -1)     # Bob  -2
q.accept(a1)            # Grace +15 bonus

try:
    vote(q, ada, +1)    # blocked: self-vote
except ValueError as e:
    print('blocked self-vote:', e)

for u in (ada, grace, bob):
    print(f'{u.name:5s} rep={u.reputation}')
print(f'scores q={q.score} a1={a1.score} a2={a2.score} accepted_id={q.accepted.id}')


blocked self-vote: cannot vote on your own post
Ada   rep=5
Grace rep=35
Bob   rep=-2
scores q=1 a1=2 a2=-1 accepted_id=2


### What got better

- `Post.score` is a **property** — no more `sum(votes)` at every call-site.
- `vote()` has one place that enforces **no self-voting** and the correct rep math.
- `Question.accept()` checks the answer really belongs to the question.
- Each class is tiny → unit-testable in isolation.

But we can still be tripped up by magic numbers (`+1/-1`, `+5/+10`), untyped strings
(status, vote direction), and no way to close a question. Let's polish.


## 🌟 Version 3 — Best: enums, statuses, constants

Small touches that make the code **easy to read, hard to misuse, easy to extend**:

- `VoteType` enum — no more magic `+1 / -1`.
- `QuestionStatus` enum — open / closed / deleted.
- Reputation constants pulled to the top so a designer can tweak them in one place.
- Accepting an answer is a one-way door (already guarded).
- Closed questions reject new answers and new votes.


In [3]:
from __future__ import annotations
from dataclasses import dataclass
from enum import Enum
from typing import Optional, List
import itertools

# --- reputation policy (one place to tune) ---
REP_UPVOTE_QUESTION = 5
REP_UPVOTE_ANSWER   = 10
REP_DOWNVOTE        = -2
REP_ACCEPTED_BONUS  = 15

class VoteType(Enum):
    UP   = +1
    DOWN = -1

class QuestionStatus(Enum):
    OPEN    = 'open'
    CLOSED  = 'closed'
    DELETED = 'deleted'

@dataclass
class User:
    id: str
    name: str
    reputation: int = 0

_pid2 = itertools.count(1)

class Post:
    def __init__(self, author: User, body: str):
        self.id = next(_pid2)
        self.author = author
        self.body = body
        self.votes: dict[str, VoteType] = {}
        self.comments: list[tuple[str, str]] = []

    @property
    def score(self) -> int:
        return sum(v.value for v in self.votes.values())

    def comment(self, user: User, text: str) -> None:
        self.comments.append((user.id, text))

class Question(Post):
    def __init__(self, author, title, body, tags):
        super().__init__(author, body)
        self.title = title
        self.tags = list(tags)
        self.answers: List[Answer] = []
        self.accepted: Optional[Answer] = None
        self.status = QuestionStatus.OPEN

    def _add_answer(self, ans: Answer) -> None:
        if self.status is not QuestionStatus.OPEN:
            raise ValueError(f'cannot answer a {self.status.value} question')
        self.answers.append(ans)

    def accept(self, ans: Answer) -> None:
        if ans not in self.answers:
            raise ValueError('not an answer to this question')
        if self.accepted is not None:
            raise ValueError('an answer is already accepted')
        self.accepted = ans
        ans.author.reputation += REP_ACCEPTED_BONUS

    def close(self) -> None:
        self.status = QuestionStatus.CLOSED

class Answer(Post):
    def __init__(self, author, question: Question, body):
        super().__init__(author, body)
        self.question = question
        question._add_answer(self)

def vote(post: Post, voter: User, v: VoteType) -> None:
    if isinstance(post, Question) and post.status is not QuestionStatus.OPEN:
        raise ValueError('question is not open for voting')
    if isinstance(post, Answer) and post.question.status is not QuestionStatus.OPEN:
        raise ValueError('parent question is not open for voting')
    if voter.id == post.author.id:
        raise ValueError('cannot vote on your own post')
    prev_vote = post.votes.get(voter.id)
    prev_val  = prev_vote.value if prev_vote is not None else 0
    post.votes[voter.id] = v
    delta = v.value - prev_val
    per_up = REP_UPVOTE_ANSWER if isinstance(post, Answer) else REP_UPVOTE_QUESTION
    if delta > 0:
        post.author.reputation += per_up * delta
    elif delta < 0:
        post.author.reputation += REP_DOWNVOTE * (-delta)

# ---- walk-through ----
ada   = User('u1','Ada')
grace = User('u2','Grace')
bob   = User('u3','Bob')

q  = Question(ada, 'What is OOD?', 'I want to learn.', ['ood','design'])
a1 = Answer(grace, q, 'Start with SOLID principles ...')
a2 = Answer(bob,   q, 'Practice with classic problems.')

vote(q,  grace, VoteType.UP)
vote(a1, ada,   VoteType.UP)
vote(a1, bob,   VoteType.UP)
vote(a2, ada,   VoteType.DOWN)

# Change-of-mind: Ada flips her down-vote on a2 to an up-vote.
vote(a2, ada,   VoteType.UP)

q.accept(a1)
q.close()

try:
    Answer(bob, q, 'late answer')
except ValueError as e:
    print('closed question rejected new answer:', e)

for u in (ada, grace, bob):
    print(f'{u.name:5s} rep={u.reputation}')
print(f'scores q={q.score} a1={a1.score} a2={a2.score} status={q.status.value}')


closed question rejected new answer: cannot answer a closed question
Ada   rep=5
Grace rep=35
Bob   rep=18
scores q=1 a1=2 a2=1 status=closed


### Why this is the "best" version (for a 1-hour interview)

- **Readable** — `VoteType.UP` reads like English; `+1` doesn't.
- **Hard to misuse** — closing a question actually *stops* new answers & votes.
- **One knob to tweak** — all rep numbers sit at the top as constants; a designer can change scoring without touching logic.
- **Easy to extend** — `QuestionStatus` already has a `DELETED` member we'll use in notebook 3.

> 🧭 **How to talk about this in an interview:**
> Present the "good" design first (proper classes). Mention the "best" touches as *iterations you'd make
> given more time*. Interviewers love seeing "I know there's a cleaner version, and here's how I'd get there."


## 🧪 Tiny self-check

Run this cell — all assertions should pass silently. If one fails, re-read the cell above and fix the bug.


In [4]:
# Fresh users so earlier reputation doesn't pollute the asserts.
u1 = User('t1','T1')
u2 = User('t2','T2')
q  = Question(u1, 'Q?', 'body', ['x'])
a  = Answer(u2, q, 'A.')

# self-vote is blocked
try:
    vote(q, u1, VoteType.UP); raise AssertionError('self-vote should raise')
except ValueError:
    pass

# up-vote on answer gives +10 to the answer's author
vote(a, u1, VoteType.UP)
assert u2.reputation == 10, u2.reputation

# flipping up -> down:  prev = +1, new = -1, delta = -2  => rep += -2 * 2 = -4
# total rep = 10 (from the up) + (-4) = 6
vote(a, u1, VoteType.DOWN)
assert u2.reputation == 6, u2.reputation

# accepting gives +15
q.accept(a)
assert u2.reputation == 6 + REP_ACCEPTED_BONUS, u2.reputation

# can't accept twice
try:
    q.accept(a); raise AssertionError('double-accept should raise')
except ValueError:
    pass

print('all self-checks passed')


all self-checks passed


### Takeaways

- Start with **nouns and verbs** → one class per noun, behavior on the class that *owns the data*.
- Cross-cutting rules (reputation) can live in a **free function** or a **policy object** (see notebook 3).
- **Enums > magic numbers** for fixed sets of values.
- **Status** fields unlock moderation (close / delete / reopen) without new subclasses.
- Always keep a tiny smoke test nearby — even print-based — so you know the lab still works.
